In [ ]:
import base64
import json
import os
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

load_dotenv()

GOOGLE_AI_STUDIO_API_KEY = os.getenv("GOOGLE_AI_STUDIO_API_KEY")
GOOGLE_API_KEY = os.environ["GOOGLE_CUSTOM_SEARCH_API_KEY"]
GOOGLE_CX = os.environ["GOOGLE_CUSTOM_SEARCH_CX"]
GOOGLE_ENDPOINT = "https://www.googleapis.com/customsearch/v1"


In [31]:
root_dir = Path().resolve().parent
output_dir = root_dir / ".cache" / "Juicero"
output_dir.mkdir(parents=True, exist_ok=True)

In [6]:
with open(root_dir / "data" / "test" / "Juicero.json", "r") as f:
    scene_data = json.loads(f.read())

In [ ]:
scene_image_system_prompt = """# Role
あなたは、バイラルなショート動画（TikTok, Reels, YouTube Shorts）を専門とする、熟練したアートディレクター兼AIプロンプトエンジニアです。あなたの目標は、動画のシーン説明を、「シネマティックなレターボックス」スタイルの動画に最適化された、高精細な画像生成プロンプトに変換することです。

# Task
提供されたシーン説明を含むJSONスクリプトを分析します。各シーンについて以下の手順を実行してください。

1.  **Visual Prompt Analysis:**
    要求されている中心的な視覚要素、スタイル（例：「実写」、「CG」）、アクションを特定します。

2.  **Reference Evaluation & Query Generation (必須ステップ):**
    各シーンについて、外部リファレンスが必要かどうかを以下の基準で判定し、必要な場合は検索クエリを作成してください。
    * **Type A: 正確性とリアリズム (Accuracy & Realism)**
        * *判定基準:* プロンプトが現実世界の実在する製品、特定の人物、ロゴ、具体的な歴史的出来事に言及しているか？
        * *アクション:* 「はい」の場合、その外見を正確に再現するための検索クエリを作成します。

    * **Type B: 定着したイメージ (Established Imagery)**
        * *判定基準:* プロンプトが一般的な視覚的比喩（メタファー）、ステレオタイプな状況（例：「札束の山」「漫画のような冷や汗」）、あるいは特定の映画的ジャンル（例：「サイバーパンク」）に依存しているか？
        * *アクション:* 「はい」の場合、その概念を視聴者が即座に理解できる「ベタ」かつ高品質な構図を見つけるための検索クエリを作成します。

    * *判定結果:* A、Bどちらも不要な場合は「None」とします。

3.  **Prompt Generation:**
    以下の構図ルールに従いつつ、visual_promptを忠実に翻訳し、リサーチから得た洞察（A/Bの判定要素）を組み込んだ、最適化された英語の画像生成プロンプトを作成します。

# Critical Composition Rules
最終的な動画は縦型（9:16）フォーマットとなり、あなたが生成した**16:9の画像を中央に配置**し、その上下にテキストオーバーレイ用の黒帯（レターボックス）が入ります。

* **Centering:** 主要な被写体とアクションは、左右の端は見切れる可能性があるため、重要な要素を置かないでください。

* **Cinematic Style:** 特段の指定がない限り、「Photorealistic, 4k, Cinematic lighting, Shot on 35mm lens」を目指します。

# Output Format
結果を以下のjson形式で出力します。

```json
{
    "scene": Scene [ID],
    "concept": [元のvisual_promptテキスト],
    "reference_needs": [A / B / A & B / None],
    "search_query_list": [判定に基づくクエリ。Noneの場合は「なし」],
    "prompt": [英語画像生成プロンプト]
}
```
"""

In [18]:
class Scene(BaseModel):
    scene: int = Field(description="The scene number.")
    concept: str = Field(description="The original visual prompt text.")
    reference_needs: Literal["A", "B", "A & B", "None"] = Field(
        description="Reference needs: must be one of 'A', 'B', 'A & B', or 'None'."
    )
    search_query_list: list[str] = Field(
        description="Search query based on the determination. an empty list if not needed."
    )
    prompt: str = Field(description="The English image generation prompt.")


class ScenesResponse(BaseModel):
    scenes: list[Scene] = Field(description="List of processed scenes.")

### 検索クエリと画像生成プロンプトの作成


In [19]:
with genai.Client(api_key=GOOGLE_AI_STUDIO_API_KEY) as client:
    user_message = f"""シーンデータを理解して検索クエリおよび画像生成プロンプトを考えたください。
---

```json
{scene_data}
```
"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=user_message,
        config={
            "system_instruction": scene_image_system_prompt,
            "response_mime_type": "application/json",
            "response_json_schema": ScenesResponse.model_json_schema(),
        },
    )
    scenes_response = ScenesResponse.model_validate_json(response.text)

### シーンごとの画像生成


In [27]:
from typing import Any

import httpx


def search_google_images(query: str, count: int = 10) -> list[dict[str, Any]]:
    params = {
        "key": GOOGLE_API_KEY,
        "cx": GOOGLE_CX,
        "searchType": "image",
        "q": query,
        "num": count,
        "safe": "active",
        "imgType": "photo",
        "gl": "jp",  # 地域を日本に設定
        "hl": "ja",  # 言語を日本語に設定
        "lr": "lang_ja",  # 検索結果を日本語に制限
        "filter": "0",  # 重複フィルタを無効化（より多くの結果）
    }

    # 同期リクエストに変更
    response = httpx.get(GOOGLE_ENDPOINT, params=params)
    response.raise_for_status()

    items = response.json().get("items", [])
    if not items:
        return []

    result = []
    for i, item in enumerate(items):
        image = item.get("image", None)
        if not image:
            continue

        result.append(
            {
                "id": f"img_{i + 1}",
                "snippet": item.get("snippet", ""),
                "link": item.get("link", ""),
                "thumbnailLink": image.get("thumbnailLink", ""),
                "width": image.get("width", 0),
                "height": image.get("height", 0),
                "rank": i + 1,
            }
        )

    return result

In [36]:
def save_top1_image(scene_id: int, queries: list[str], count: int, save_dir: Path) -> list[Path]:
    saved_paths = []
    j_query = 0
    for query in queries:
        query_results = search_google_images(query=query, count=count)

        file_name = f"scene_{scene_id}_{j_query + 1}.jpg"
        for result in query_results:
            image_url = result["thumbnailLink"]
            image_response = httpx.get(image_url)
            if image_response.status_code == 200:
                with open(save_dir / file_name, "wb") as img_file:
                    img_file.write(image_response.content)
                saved_paths.append(save_dir / file_name)
                j_query += 1
                break
    return saved_paths

In [44]:
scene_image_dir = output_dir / "scene_images"
scene_image_dir.mkdir(parents=True, exist_ok=True)

In [46]:
import time

In [ ]:
for i in range(len(scenes_response.scenes)):
    scene = scenes_response.scenes[i]

    if scene.search_query_list:
        reference_image_dir = output_dir / "reference_images" / f"scene_{scene.scene}"
        reference_image_dir.mkdir(parents=True, exist_ok=True)

        reference_images = save_top1_image(
            scene_id=scene.scene, queries=scene.search_query_list, count=5, save_dir=reference_image_dir
        )
    else:
        reference_images = []

    reference_image_b64s = []
    for ref_image_path in reference_images:
        with open(ref_image_path, "rb") as f:
            ref_image_bytes = f.read()
            ref_image_b64 = base64.b64encode(ref_image_bytes).decode()
            reference_image_b64s.append(ref_image_b64)

    with genai.Client(api_key=GOOGLE_AI_STUDIO_API_KEY) as client:
        edit_response = client.models.generate_content(
            model="gemini-3-pro-image-preview",
            contents=[{"text": scene.prompt}]
            + [
                {"inline_data": {"mime_type": "image/png", "data": reference_image_b64}}
                for reference_image_b64 in reference_image_b64s
            ],
            config=types.GenerateContentConfig(image_config=types.ImageConfig(aspect_ratio="16:9", image_size="2K")),
        )

        # 編集後画像を保存
        image_parts = [part for part in edit_response.parts if part.inline_data]
        if image_parts:
            edited_image = image_parts[0].as_image()
            edited_path = scene_image_dir / f"scene_{scene.scene}.png"
            edited_image.save(edited_path)
            print(f"✅ 編集画像を保存: {edited_path}")
        else:
            print("❌ 画像編集に失敗しました。")

    time.sleep(5)

✅ 編集画像を保存: /Users/yamaguchiteppei/opt/post/weekly_dev/sns_ai_automation_agency/.cache/Juicero/scene_images/scene_1.png
✅ 編集画像を保存: /Users/yamaguchiteppei/opt/post/weekly_dev/sns_ai_automation_agency/.cache/Juicero/scene_images/scene_2.png
✅ 編集画像を保存: /Users/yamaguchiteppei/opt/post/weekly_dev/sns_ai_automation_agency/.cache/Juicero/scene_images/scene_3.png
✅ 編集画像を保存: /Users/yamaguchiteppei/opt/post/weekly_dev/sns_ai_automation_agency/.cache/Juicero/scene_images/scene_4.png
✅ 編集画像を保存: /Users/yamaguchiteppei/opt/post/weekly_dev/sns_ai_automation_agency/.cache/Juicero/scene_images/scene_5.png
✅ 編集画像を保存: /Users/yamaguchiteppei/opt/post/weekly_dev/sns_ai_automation_agency/.cache/Juicero/scene_images/scene_6.png
✅ 編集画像を保存: /Users/yamaguchiteppei/opt/post/weekly_dev/sns_ai_automation_agency/.cache/Juicero/scene_images/scene_7.png
✅ 編集画像を保存: /Users/yamaguchiteppei/opt/post/weekly_dev/sns_ai_automation_agency/.cache/Juicero/scene_images/scene_8.png


/Users/yamaguchiteppei/.local/share/uv/python/cpython-3.12.9-macos-aarch64-none/lib/python3.12/ast.py:280: RuntimeWarning: coroutine 'search_google_images_async' was never awaited
  yield item


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-3-pro-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-3-pro-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-3-pro-image\nPlease retry in 32.597048093s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'

In [20]:
scenes_response.scenes[0]

Scene(scene=1, concept='【実写/寄り】白くて洗練されたデザインの「Juicero」マシンが、高級キッチンのカウンターに鎮座している。ライティングはApple製品CM風。', reference_needs='A & B', search_query_list=['Juicero machine design', 'Apple product commercial style lighting', 'luxury kitchen interior design'], prompt='Close-up of a sleek, white Juicero machine, impeccably designed, sitting on a pristine counter in a luxurious, modern kitchen. The lighting is bright and clean, mimicking the high-key, sophisticated aesthetic of an Apple product commercial. Photorealistic, 8k, Cinematic lighting, Shot on 35mm lens.')

In [12]:
print(response.text)

```json
[
    {
        "scene": 1,
        "concept": "【実写/寄り】白くて洗練されたデザインの「Juicero」マシンが、高級キッチンのカウンターに鎮座している。ライティングはApple製品CM風。",
        "reference_needs": "A & B",
        "search_query": "Juicero machine real life, Apple product commercial kitchen lighting",
        "prompt": "Close-up shot of a white, sleekly designed Juicero machine, prominently placed on a high-end kitchen counter. The lighting is bright, clean, and minimalist, mimicking an Apple product commercial, emphasizing its refined aesthetic. Photorealistic, 8k, Cinematic lighting, Shot on 35mm lens."
    },
    {
        "scene": 2,
        "concept": "【実写/分割画面】左画面でマシンがゆっくり稼働開始。右画面では女性記者がシンクでパックを両手で鷲掴みにしている。",
        "reference_needs": "B",
        "search_query": "Juicero pack real life, split screen comparison, person squeezing juice pouch",
        "prompt": "Split screen view. On the left side, a Juicero machine slowly begins to operate, with subtle light indicators. On the right side, a female reporter-type figure

In [7]:
scene_data

{'video_title': '130億円がゴミに？シリコンバレー最大の「意識高い系」失敗伝説',
 'scenes': [{'scene_id': 1,
   'duration': '2.0s',
   'text_content': '4万円の最新ハイテクジューサー',
   'visual_prompt': '【実写/寄り】白くて洗練されたデザインの「Juicero」マシンが、高級キッチンのカウンターに鎮座している。ライティングはApple製品CM風。',
   'rationale': '修正なし'},
  {'scene_id': 2,
   'duration': '1.5s',
   'text_content': 'VS 人間の素手',
   'visual_prompt': '【実写/分割画面】左画面でマシンがゆっくり稼働開始。右画面では女性記者がシンクでパックを両手で鷲掴みにしている。',
   'rationale': '修正なし'},
  {'scene_id': 3,
   'duration': '2.0s',
   'text_content': 'まさかの人間圧勝で草',
   'visual_prompt': '【実写/右側強調】右画面の女性の手からドバドバとジュースが出る。左のマシンはまだチョロチョロと滴っているだけの比較。',
   'rationale': '修正なし'},
  {'scene_id': 4,
   'duration': '2.0s',
   'text_content': 'ただの手抜き商品に見えるが…',
   'visual_prompt': '【CG/展開図】マシンが空中で爆発するように分解され、無数の部品が広がるアニメーション。背景は黒。',
   'rationale': '【論理接続】前のシーンで「敗北」を見せられた視聴者は「これはゴミだ」と直感している。いきなり「詐欺じゃない」と否定せず、「そう見えますが」と一度感情を受け止めてから逆接（Turn）へ繋ぐことで認知摩擦を消す。'},
  {'scene_id': 5,
   'duration': '1.5s',
   'text_content': '中身はムダに変態技術の塊',
   'visual_prompt': '【実写/マクロ】